In [1]:
import pandas as pd
import numpy as np

In [2]:
DS = pd.read_excel("2_LabelwithDeepSeek.xlsx")
GPT = pd.read_excel("2_LabelwithOpenAI_4o.xlsx")
GEMINI = pd.read_excel("2_LabelwithGemini.xlsx")
GEMINI.head()

,sentence,label
0,"In December, significant increases in the pric...",hawkish
1,"On the other hand, fruit and vegetable prices,...",dovish
2,"With 7.72 percent, annual inflation was realiz...",neutral
3,"VAT cuts made in food, health and education se...",dovish
4,Even if the downward course in the inflation t...,hawkish


In [3]:
AllModels = pd.DataFrame(columns=["Sentences", "gpt", "gemini", "deepseek"])
AllModels.Sentences = GPT.sentence
AllModels.gpt = GPT.label
AllModels.gemini = GEMINI.label
AllModels.deepseek = DS.label
AllModels.head()

,Sentences,gpt,gemini,deepseek
0,"In December, significant increases in the pric...",hawkish,hawkish,hawkish
1,"On the other hand, fruit and vegetable prices,...",neutral,dovish,neutral
2,"With 7.72 percent, annual inflation was realiz...",dovish,neutral,neutral
3,"VAT cuts made in food, health and education se...",neutral,dovish,neutral
4,Even if the downward course in the inflation t...,neutral,hawkish,neutral


In [4]:
def majority_vote(row):
    votes = [row["gpt"], row["gemini"], row["deepseek"]]
    counts = pd.Series(votes).value_counts()

    # En yüksek oy sayısı 1 ise → herkes farklı
    if counts.iloc[0] == 1:
        return "NoVote"
    else:
        return counts.idxmax()

AllModels["majority_voting"] = AllModels.apply(majority_vote, axis=1)

AllModels.head()

,Sentences,gpt,gemini,deepseek,majority_voting
0,"In December, significant increases in the pric...",hawkish,hawkish,hawkish,hawkish
1,"On the other hand, fruit and vegetable prices,...",neutral,dovish,neutral,neutral
2,"With 7.72 percent, annual inflation was realiz...",dovish,neutral,neutral,neutral
3,"VAT cuts made in food, health and education se...",neutral,dovish,neutral,neutral
4,Even if the downward course in the inflation t...,neutral,hawkish,neutral,neutral


## Labelling Process and Krispendorf Alpha Scores

In [8]:
AllModels.gpt.value_counts()

gpt
neutral    10513
hawkish     3722
dovish      2582
Name: count, dtype: int64

In [6]:
import pandas as pd
import numpy as np

# Majority Vote
AllModels["majority_vote"] = (
    AllModels[["gpt", "gemini", "deepseek"]]
    .mode(axis=1)[0]
)

# Joint Decision (3 model tamamen aynıysa)
AllModels["joint_decision"] = np.where(
    (AllModels["gpt"] == AllModels["gemini"]) &
    (AllModels["gpt"] == AllModels["deepseek"]),
    AllModels["gpt"],
    np.nan
)

In [7]:
labels = ["neutral", "hawkish", "dovish"]

def label_counts(col):
    return AllModels[col].value_counts().reindex(labels, fill_value=0)

label_table = pd.DataFrame({
    "GPT": label_counts("gpt"),
    "GEMINI": label_counts("gemini"),
    "DeepSeek": label_counts("deepseek"),
    "Majority Vote": label_counts("majority_vote"),
    "Joint Decision": label_counts("joint_decision")
}).T

label_table

,neutral,hawkish,dovish
GPT,10513,3722,2582
GEMINI,3861,7912,5044
DeepSeek,13000,2923,894
Majority Vote,10326,3971,2520
Joint Decision,3756,2556,872


In [9]:
import krippendorff

# Stringleri sayıya çeviriyoruz
label_map = {"neutral": 0, "hawkish": 1, "dovish": 2}

alpha_df = AllModels[["gpt", "gemini", "deepseek", "majority_vote"]].replace(label_map)

def compute_alpha(col1, col2):
    data = np.array([alpha_df[col1], alpha_df[col2]])
    return krippendorff.alpha(data, level_of_measurement='nominal')

alpha_table = pd.DataFrame(index=["GPT","GEMINI","DeepSeek","Majority Vote"],
                           columns=["GPT","GEMINI","DeepSeek","Majority Vote"])

for i in alpha_table.index:
    for j in alpha_table.columns:
        if i != j:
            alpha_table.loc[i,j] = compute_alpha(i.lower().replace(" ","_"),
                                                 j.lower().replace(" ","_"))
        else:
            alpha_table.loc[i,j] = "-"
            
alpha_table

C:\Users\muham\AppData\Local\Temp\ipykernel_22188\566389572.py:6: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  alpha_df = AllModels[["gpt", "gemini", "deepseek", "majority_vote"]].replace(label_map)


,GPT,GEMINI,DeepSeek,Majority Vote
GPT,-,0.350919,0.595615,0.95327
GEMINI,0.350919,-,0.104697,0.390284
DeepSeek,0.595615,0.104697,-,0.65014
Majority Vote,0.95327,0.390284,0.65014,-


In [23]:
ForDateDf = pd.read_excel(r"C:\Users\muham\Desktop\CBRT-LLM Revision\1-ExtractReports\1_SummaryReportsV2.xlsx")
ForDateDf.head()

,Unnamed: 0,ReportName,Sentences
0,0,Summary_2006_1_January.pdf,"In December, significant increases in the pric..."
1,1,Summary_2006_1_January.pdf,"On the other hand, fruit and vegetable prices,..."
2,2,Summary_2006_1_January.pdf,"With 7.72 percent, annual inflation was realiz..."
3,3,Summary_2006_1_January.pdf,"VAT cuts made in food, health and education se..."
4,4,Summary_2006_1_January.pdf,Even if the downward course in the inflation t...


In [24]:
AllModels2 = AllModels.copy()
ForDateDf2 = ForDateDf.copy()

AllModels2["Sentences_key"] = AllModels2["Sentences"].astype(str).str.strip()
ForDateDf2["Sentences_key"] = ForDateDf2["Sentences"].astype(str).str.strip()
sent_to_date = (
    ForDateDf2
    .dropna(subset=["Sentences_key", "ReportName"])
    .drop_duplicates(subset=["Sentences_key"], keep="first")
    .set_index("Sentences_key")["ReportName"]
)
AllModels2.head()

AllModels2["Date"] = AllModels2["Sentences_key"].map(sent_to_date)

AllModels = (
    AllModels2
    .drop(columns=["Sentences_key"])
)

# Eşleşmeyen var mı?
print("Date atanamayan satır sayısı:", AllModels["Date"].isna().sum())

AllModels.head()


Date atanamayan satır sayısı: 0


,Sentences,gpt,gemini,deepseek,majority_voting,Date
0,"In December, significant increases in the pric...",hawkish,hawkish,hawkish,hawkish,Summary_2006_1_January.pdf
1,"On the other hand, fruit and vegetable prices,...",neutral,dovish,neutral,neutral,Summary_2006_1_January.pdf
2,"With 7.72 percent, annual inflation was realiz...",dovish,neutral,neutral,neutral,Summary_2006_1_January.pdf
3,"VAT cuts made in food, health and education se...",neutral,dovish,neutral,neutral,Summary_2006_1_January.pdf
4,Even if the downward course in the inflation t...,neutral,hawkish,neutral,neutral,Summary_2006_1_January.pdf


In [25]:
AllModels2["Date"] = AllModels2["Sentences_key"].map(sent_to_date)

In [26]:
AllModels = (
    AllModels2
    .drop(columns=["Sentences_key"])
)

# Eşleşmeyen var mı?
print("Date atanamayan satır sayısı:", AllModels["Date"].isna().sum())

AllModels.head()


Date atanamayan satır sayısı: 0


,Sentences,gpt,gemini,deepseek,majority_voting,Date
0,"In December, significant increases in the pric...",hawkish,hawkish,hawkish,hawkish,Summary_2006_1_January.pdf
1,"On the other hand, fruit and vegetable prices,...",neutral,dovish,neutral,neutral,Summary_2006_1_January.pdf
2,"With 7.72 percent, annual inflation was realiz...",dovish,neutral,neutral,neutral,Summary_2006_1_January.pdf
3,"VAT cuts made in food, health and education se...",neutral,dovish,neutral,neutral,Summary_2006_1_January.pdf
4,Even if the downward course in the inflation t...,neutral,hawkish,neutral,neutral,Summary_2006_1_January.pdf


In [28]:
AllModels["Date"] = (
    AllModels["Date"]
    .str.replace(r"Summary_", "", regex=True)
    .str.replace(r"_\d+_", " - ", regex=True)
    .str.replace(r"\.pdf$", "", regex=True)
)

AllModels.head()

,Sentences,gpt,gemini,deepseek,majority_voting,Date
0,"In December, significant increases in the pric...",hawkish,hawkish,hawkish,hawkish,2006 - January
1,"On the other hand, fruit and vegetable prices,...",neutral,dovish,neutral,neutral,2006 - January
2,"With 7.72 percent, annual inflation was realiz...",dovish,neutral,neutral,neutral,2006 - January
3,"VAT cuts made in food, health and education se...",neutral,dovish,neutral,neutral,2006 - January
4,Even if the downward course in the inflation t...,neutral,hawkish,neutral,neutral,2006 - January


In [29]:
AllModels.tail()

,Sentences,gpt,gemini,deepseek,majority_voting,Date
16812,Leading indicators suggest milder price increa...,dovish,dovish,neutral,dovish,2025 - December
16813,Underlying inflation indicators are also expec...,dovish,dovish,dovish,dovish,2025 - December
16814,"Against this backdrop, leading indicators impl...",dovish,dovish,dovish,dovish,2025 - December
16815,Committee (the Committee) has decided to reduc...,dovish,dovish,dovish,dovish,2025 - December
16816,The Committee has also lowered the Central Ban...,dovish,dovish,dovish,dovish,2025 - December


In [34]:
NoVoteDf = AllModels[AllModels["majority_voting"] == "NoVote"]
print(len(NoVoteDf))
NoVoteDf.head()

169


,Sentences,gpt,gemini,deepseek,majority_voting,Date
42,Declining interests and extending maturities i...,dovish,hawkish,neutral,NoVote,2006 - January
360,"Prices of unprocessed foods, which usually dec...",dovish,hawkish,neutral,NoVote,2006 - June
380,It is expected that the slowdown in domestic d...,neutral,dovish,hawkish,NoVote,2006 - June
392,Even the high inflation figures in April did n...,hawkish,dovish,neutral,NoVote,2006 - June
478,"Moreover, the expanding tendency in labor supp...",neutral,dovish,hawkish,NoVote,2006 - June


In [33]:
NoVoteDf.to_excel("3_NoVoteSentences.xlsx")

In [35]:
with open("3_NoVoteSentences.txt", "w", encoding="utf-8") as f:
    f.write("\n".join(NoVoteDf["Sentences"].astype(str)))

In [37]:
HumanVotingDf = pd.read_excel("HumanVotingNoVote.xlsx")
HumanVotingDf.head()

,Unnamed: 0,Sentences,gpt,gemini,deepseek,majority_voting,Date,HumanVoting,RES
0,42,Declining interests and extending maturities i...,dovish,hawkish,neutral,NoVote,2006 - January,dovish,True
1,360,"Prices of unprocessed foods, which usually dec...",dovish,hawkish,neutral,NoVote,2006 - June,dovish,True
2,380,It is expected that the slowdown in domestic d...,neutral,dovish,hawkish,NoVote,2006 - June,hawkish,False
3,392,Even the high inflation figures in April did n...,hawkish,dovish,neutral,NoVote,2006 - June,hawkish,True
4,478,"Moreover, the expanding tendency in labor supp...",neutral,dovish,hawkish,NoVote,2006 - June,dovish,False


In [36]:
AllModels.head()

,Sentences,gpt,gemini,deepseek,majority_voting,Date
0,"In December, significant increases in the pric...",hawkish,hawkish,hawkish,hawkish,2006 - January
1,"On the other hand, fruit and vegetable prices,...",neutral,dovish,neutral,neutral,2006 - January
2,"With 7.72 percent, annual inflation was realiz...",dovish,neutral,neutral,neutral,2006 - January
3,"VAT cuts made in food, health and education se...",neutral,dovish,neutral,neutral,2006 - January
4,Even if the downward course in the inflation t...,neutral,hawkish,neutral,neutral,2006 - January


In [38]:
# 1) Sentences -> HumanVoting mapping (169 cümle)
hv_map = (HumanVotingDf
          .dropna(subset=["Sentences", "HumanVoting"])
          .drop_duplicates(subset=["Sentences"], keep="first")
          .set_index("Sentences")["HumanVoting"])

# 2) AllModels içinde eşleşen satırları bul
mask = AllModels["Sentences"].isin(hv_map.index)

print("Eşleşen satır sayısı:", mask.sum())

# 3) majority_voting'i human voting ile overwrite et
AllModels.loc[mask, "majority_voting"] = AllModels.loc[mask, "Sentences"].map(hv_map)

AllModels.head()


Eşleşen satır sayısı: 169


,Sentences,gpt,gemini,deepseek,majority_voting,Date
0,"In December, significant increases in the pric...",hawkish,hawkish,hawkish,hawkish,2006 - January
1,"On the other hand, fruit and vegetable prices,...",neutral,dovish,neutral,neutral,2006 - January
2,"With 7.72 percent, annual inflation was realiz...",dovish,neutral,neutral,neutral,2006 - January
3,"VAT cuts made in food, health and education se...",neutral,dovish,neutral,neutral,2006 - January
4,Even if the downward course in the inflation t...,neutral,hawkish,neutral,neutral,2006 - January


In [39]:
AllModels["majority_voting"].value_counts()

majority_voting
neutral    10333
hawkish     4025
dovish      2459
Name: count, dtype: int64

In [43]:
AllModelsFinal = AllModels[["Sentences", "majority_voting", "Date"]].rename(columns={"majority_voting": "label"})
AllModelsFinal.head()

,Sentences,label,Date
0,"In December, significant increases in the pric...",hawkish,2006 - January
1,"On the other hand, fruit and vegetable prices,...",neutral,2006 - January
2,"With 7.72 percent, annual inflation was realiz...",neutral,2006 - January
3,"VAT cuts made in food, health and education se...",neutral,2006 - January
4,Even if the downward course in the inflation t...,neutral,2006 - January


In [44]:
AllModelsFinal.to_excel("5_FinalDataset.xlsx")

In [45]:
AllModelsFinal.value_counts("label")

label
neutral    10333
hawkish     4025
dovish      2459
Name: count, dtype: int64